# GEDI Canopy Height Pipeline

This notebook processes NASA GEDI Level 2A HDF5 files stored in S3 through four sequential phases:

| Phase | Description |
|---|---|
| **Phase 1** | High-performance batch extraction of GEDI shots from S3 |
| **Phase 1a** | Year parsing from GEDI filenames |
| **Phase 1b** | Harmonization to the SMAP 9 km grid |
| **Phase 1c** | Spatial join to assign shots to study-area jurisdictions |

**Source:** `s3://central-virginia-tree-canopy-project/GEDI/GEDI02_A/002/`  
**Final Summary output:** `s3://central-virginia-tree-canopy-project/gedi-county-summary/virginia_gedi_county_summary.csv`
**Final Detailed output:** `s3://central-virginia-tree-canopy-project/gedi-county-detailed/virginia_gedi_county_canopy_height.csv`

## Cell 1 — Install Dependencies

In [1]:
import subprocess, sys

# Pin the entire boto stack atomically — all four packages must be compatible
boto_stack = [
    "botocore==1.43.0",
    "boto3==1.43.0",
    "s3fs==2026.6.0",
]

# Install boto stack with --no-deps to prevent pip from overriding the pins
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "--force-reinstall"] + boto_stack)

# Install all other dependencies normally
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                       "h5py", "geopandas", "xarray", "netCDF4", "pyarrow"])

print("Boto Stack and all dependencies installed.")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
awscli 1.45.42 requires botocore==1.43.42, but you have botocore 1.43.0 which is incompatible.
awscli 1.45.42 requires s3transfer<0.20.0,>=0.19.0, but you have s3transfer 0.17.1 which is incompatible.
sagemaker 2.257.3 requires attrs<26,>=24, but you have attrs 26.1.0 which is incompatible.


Boto Stack and all dependencies installed.


## Cell 2 — Imports

In [2]:
import os
import time
import io
import re
import urllib.request

from concurrent.futures import ThreadPoolExecutor, as_completed

import statistics

import boto3
import s3fs
import h5py
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr

print("Imports successful.")

Imports successful.


## Cell 3 — Configuration

Edit this cell to change S3 paths, bounding box, grid resolution, or target jurisdictions.

In [3]:
# ── S3 settings ───────────────────────────────────────────────────────────────
S3_BUCKET             = "central-virginia-tree-canopy-project"
S3_PREFIX             = "GEDI/GEDI02_A/002/"     
GEDI_COUNTY_S3_PREFIX = "gedi-county-summary/"
GEDI_DETAILED_COUNTY_S3_PREFIX = "gedi-county-detailed/"

# ── Local output directory ────────────────────────────────────────────────────
OUTPUT_DIR        = "/home/ec2-user/SageMaker/gedi_output"
OUTPUT_PARQUET    = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_multiyear.parquet")
OUTPUT_NETCDF     = os.path.join(OUTPUT_DIR, "virginia_gedi_canopy_grid.nc")
OUTPUT_COUNTY_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_county_summary.csv")
OUTPUT_DETAILED_COUNTY_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_county_canopy_height.csv")
OUTPUT_DETAILED_COUNTY_CANOPY_HEIGHT_CSV = os.path.join(OUTPUT_DIR, "virginia_gedi_county_canopy_height.csv")

# ── Spatial bounds (Virginia 8-jurisdiction study area) ───────────────────────
MIN_LON, MIN_LAT, MAX_LON, MAX_LAT = -79.1721, 37.3296, -77.6873, 38.4755

# ── SMAP grid resolution (~9 km) ─────────────────────────────────────────────
GRID_RES = 0.081

# ── Target jurisdictions (Name, State FIPS, County/Place FIPS, Type) ─────────
TARGET_JURISDICTIONS = [
    ("Albemarle",       "51", "003",   "county"),
    ("Augusta",         "51", "015",   "county"),
    ("Buckingham",      "51", "029",   "county"),
    ("Charlottesville", "51", "14968", "place"),
    ("Fluvanna",        "51", "065",   "county"),
    ("Greene",          "51", "079",   "county"),
    ("Louisa",          "51", "109",   "county"),
    ("Nelson",          "51", "125",   "county"),
    ("Orange",          "51", "137",   "county"),
    ("Rockingham",      "51", "165",   "county"),
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Configuration loaded.")
print(f"  GEDI source  : s3://{S3_BUCKET}/{S3_PREFIX}")
print(f"  Output dir   : {OUTPUT_DIR}")
print(f"  S3 output    : s3://{S3_BUCKET}/{GEDI_COUNTY_S3_PREFIX}")

Configuration loaded.
  GEDI source  : s3://central-virginia-tree-canopy-project/GEDI/GEDI02_A/002/
  Output dir   : /home/ec2-user/SageMaker/gedi_output
  S3 output    : s3://central-virginia-tree-canopy-project/gedi-county-summary/


## Cell 4 — Helper Functions

In [4]:
def parse_year_from_filename(filename: str):
    """Extract the year from a standard GEDI filename (e.g., GEDI02_A_2022143...)."""
    year_match = re.search(r'GEDI02_A_(\d{4})', filename)
    if year_match:
        return int(year_match.group(1))
    return None


def fetch_boundary(name: str, state_fips: str, geo_id: str, geo_type: str) -> gpd.GeoDataFrame:
    """Fetch boundary GeoJSON directly from the US Census TIGERweb API."""
    if geo_type == "place":
        url = (
            "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
            "Places_CouSub_ConCity_SubMCD/MapServer/4/query"
            f"?where=STATE='{state_fips}'+AND+PLACE='{geo_id}'"
            "&outFields=NAME,STATE,PLACE&f=geojson&outSR=4326"
        )
    else:
        url = (
            "https://tigerweb.geo.census.gov/arcgis/rest/services/TIGERweb/"
            "State_County/MapServer/11/query"
            f"?where=STATE='{state_fips}'+AND+COUNTY='{geo_id}'"
            "&outFields=NAME,STATE,COUNTY&f=geojson&outSR=4326"
        )
    print(f"  Fetching boundary for {name}...")
    with urllib.request.urlopen(url, timeout=20) as r:
        gdf = gpd.read_file(r)
    if gdf.empty:
        raise ValueError(f"No boundary found for {name} (FIPS {state_fips}/{geo_id})")
    gdf = gdf.set_crs("EPSG:4326")
    gdf['jurisdiction'] = name
    return gdf

def process_one_file(key):
    file_name = os.path.basename(key)
    results = []

    year = parse_year_from_filename(file_name)
    if not year:
        print(f"Warning: Could not parse year from {file_name}. Skipping.")
        return results

    try:
        s3_path = f"s3://{S3_BUCKET}/{key}"
        with fs.open(s3_path, "rb") as f:
            raw_bytes = f.read()

        with h5py.File(io.BytesIO(raw_bytes), "r") as hf:
            beams = [k for k in hf.keys() if k.startswith("BEAM")]
            for beam in beams:
                if "lat_lowestmode" not in hf[beam]:
                    continue

                lats = hf[f"{beam}/lat_lowestmode"][:]
                lons = hf[f"{beam}/lon_lowestmode"][:]
                spatial_mask = (
                    (lons >= MIN_LON) & (lons <= MAX_LON) &
                    (lats >= MIN_LAT) & (lats <= MAX_LAT)
                )
                if not np.any(spatial_mask):
                    continue

                # Cheap 1D reads first -- quality_flag/sensitivity are single
                # columns, far cheaper than rh's 101-column 2D array.
                quality     = hf[f"{beam}/quality_flag"][:][spatial_mask]
                sensitivity = hf[f"{beam}/sensitivity"][:][spatial_mask]
                quality_ok  = (quality == 1) & (sensitivity > 0.9)

                # Only in the rare case where NO spatially-masked point would
                # survive quality filtering anyway does this actually skip
                # work -- reordering alone doesn't reduce rh's chunk-read
                # cost once we do need it, since HDF5 chunks aren't aligned
                # to the mask.
                if not np.any(quality_ok):
                    continue

                rh98 = hf[f"{beam}/rh"][:, 98][spatial_mask]

                # Build the fully-combined mask once, then construct the
                # DataFrame directly from already-filtered arrays -- avoids
                # materializing a larger intermediate DataFrame that then
                # gets filtered a second time.
                rh_ok = (rh98 > 0) & (rh98 < 100)
                final_mask = quality_ok & rh_ok
                if not np.any(final_mask):
                    continue

                valid_df = pd.DataFrame({
                    "longitude":    lons[spatial_mask][final_mask],
                    "latitude":     lats[spatial_mask][final_mask],
                    "quality_flag": quality[final_mask],
                    "sensitivity":  sensitivity[final_mask],
                    "rh98":         rh98[final_mask],
                    "year":         year,
                    "file_source":  file_name,
                    "beam":         beam
                })
                if not valid_df.empty:
                    results.append(valid_df)

    except Exception as e:
        print(f"Error reading {file_name}: {e}")

    return results

print("Helper functions defined.")

Helper functions defined.


## Cell 5 — Phase 1 & 1a: Discover GEDI Files in S3

In [5]:
print(f"Scanning s3://{S3_BUCKET}/{S3_PREFIX} for GEDI HDF5 files...")

s3 = boto3.client("s3")
paginator = s3.get_paginator("list_objects_v2")

h5_keys = []
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_PREFIX):
    for obj in page.get("Contents", []):
        if obj["Key"].endswith(".h5"):
            h5_keys.append(obj["Key"])

print(f"Found {len(h5_keys)} GEDI HDF5 files.")
if h5_keys:
    print(f"  First : {h5_keys[0]}")
    print(f"  Last  : {h5_keys[-1]}")

Scanning s3://central-virginia-tree-canopy-project/GEDI/GEDI02_A/002/ for GEDI HDF5 files...
Found 398 GEDI HDF5 files.
  First : GEDI/GEDI02_A/002/GEDI02_A_2019113174925_O02048_03_T02621_02_003_01_V002.h5
  Last  : GEDI/GEDI02_A/002/GEDI02_A_2025189013841_O37209_02_T08829_02_004_02_V002.h5


## Cell 6 — Phase 1 & 1a: Batch Extraction with Spatial Masking and Quality Filtering

In [6]:
# Batch extraction helper function

def extract_gedi_dataframes(h5_keys, max_workers=16):
    """
    Concurrently download and process GEDI .h5 granules from S3, returning
    a combined list of per-beam DataFrames that passed spatial/quality filtering.

    Parameters
    ----------
    h5_keys : list[str]
        S3 object keys for the .h5 granules to process.
    max_workers : int, optional
        Number of concurrent threads to use for downloading/processing files
        (default: 16). Network I/O bound, so pushing this higher is usually
        safe until you hit S3 throttling or bandwidth limits.

    Returns
    -------
    list[pd.DataFrame]
        List of per-beam DataFrames, one per beam that had valid data after
        spatial masking and quality filtering.
    """
    all_dfs = []
    cell_start_time = time.time()
    completed = 0

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_one_file, key): key for key in h5_keys}
        for future in as_completed(futures):
            key = futures[future]
            try:
                file_dfs = future.result()
                all_dfs.extend(file_dfs)
            except Exception as e:
                print(f"Unhandled error processing {os.path.basename(key)}: {e}")
            completed += 1
            if completed % 10 == 0:
                elapsed = time.time() - cell_start_time
                print(f"Processed {completed}/{len(h5_keys)} files... "
                      f"({elapsed:.2f}s elapsed total)")

    total_elapsed = time.time() - cell_start_time
    minutes, seconds = divmod(total_elapsed, 60)
    print(f"\nExtraction complete. Beams with valid data: {len(all_dfs)}")
    print(f"🚀 Total execution time: {int(minutes)}m {seconds:.2f}s")

    return all_dfs

print("Batch extraction helper function defined.")

Batch extraction helper function defined.


In [ ]:
fs = s3fs.S3FileSystem(anon=False)

all_dfs = extract_gedi_dataframes(h5_keys, max_workers=9)  #Max Workers can be set to 16 or 24 
# all_dfs = []

# # This is a long running processing....let's time it
# cell_start_time = time.time()

# for i, key in enumerate(h5_keys):
#     file_start_time = time.time()
#     file_name = os.path.basename(key)

#     # Phase 1a: Parse year from filename
#     year = parse_year_from_filename(file_name)
#     if not year:
#         print(f"Warning: Could not parse year from {file_name}. Skipping.")
#         continue

#     try:
#         # Stream HDF5 file from S3 into memory
#         s3_path = f"s3://{S3_BUCKET}/{key}"
#         with fs.open(s3_path, "rb") as f:
#             raw_bytes = f.read()

#         with h5py.File(io.BytesIO(raw_bytes), 'r') as hf:
#             beams = [k for k in hf.keys() if k.startswith('BEAM')]

#             for beam in beams:
#                 if 'lat_lowestmode' not in hf[beam]:
#                     continue

#                 # Step 1: Extract coordinates first for rapid spatial masking
#                 lats = hf[f"{beam}/lat_lowestmode"][:]
#                 lons = hf[f"{beam}/lon_lowestmode"][:]

#                 spatial_mask = (
#                     (lons >= MIN_LON) & (lons <= MAX_LON) &
#                     (lats >= MIN_LAT) & (lats <= MAX_LAT)
#                 )

#                 if not np.any(spatial_mask):
#                     continue

#                 # Step 2: Extract attributes only for points inside the bbox
#                 quality    = hf[f"{beam}/quality_flag"][:][spatial_mask]
#                 sensitivity = hf[f"{beam}/sensitivity"][:][spatial_mask]
#                 rh98       = hf[f"{beam}/rh"][:, 98][spatial_mask]

#                 beam_df = pd.DataFrame({
#                     'longitude':    lons[spatial_mask],
#                     'latitude':     lats[spatial_mask],
#                     'quality_flag': quality,
#                     'sensitivity':  sensitivity,
#                     'rh98':         rh98,
#                     'year':         year,
#                     'file_source':  file_name,
#                     'beam':         beam
#                 })

#                 # Step 3: Strict quality filtering
#                 valid_df = beam_df[
#                     (beam_df['quality_flag'] == 1) &
#                     (beam_df['sensitivity']  > 0.9) &
#                     (beam_df['rh98']         > 0)   &
#                     (beam_df['rh98']         < 100)
#                 ]

#                 if not valid_df.empty:
#                     all_dfs.append(valid_df)

#     except Exception as e:
#         print(f"Error reading {file_name}: {e}")

#     if (i + 1) % 10 == 0:
#         file_elapsed = time.time() - file_start_time
#         print(f"Processed {i + 1}/{len(h5_keys)} files... in {file_elapsed:.2f} seconds.")

# #Calculate and display total run time
# total_elapsed = time.time() - cell_start_time
# minutes, seconds = divmod(total_elapsed, 60)

# print(f"\nExtraction complete. Beams with valid data: {len(all_dfs)}")
# print(f"🚀 Total execution time: {int(minutes)}m {seconds:.2f}s")


Processed 10/398 files... (134.33s elapsed total)
Processed 20/398 files... (226.31s elapsed total)
Processed 30/398 files... (341.34s elapsed total)
Processed 40/398 files... (461.24s elapsed total)
Processed 50/398 files... (565.67s elapsed total)
Processed 60/398 files... (680.60s elapsed total)
Processed 70/398 files... (793.33s elapsed total)
Processed 80/398 files... (911.66s elapsed total)
Processed 90/398 files... (1010.42s elapsed total)
Processed 100/398 files... (1099.45s elapsed total)
Processed 110/398 files... (1249.82s elapsed total)


## Cell 7 — Save Extracted Points to Parquet

In [7]:
if not all_dfs:
    raise RuntimeError("No valid GEDI shots found within the bounding box. Check S3 prefix and bbox settings.")

df_gedi = pd.concat(all_dfs, ignore_index=True)
df_gedi.to_parquet(OUTPUT_PARQUET, index=False)

print(f"Total valid GEDI shots saved : {len(df_gedi):,}")
print(f"Years covered                : {sorted(df_gedi['year'].unique())}")
print(f"Parquet file                 : {OUTPUT_PARQUET}")
df_gedi.head()

Total valid GEDI shots saved : 2,059,963
Years covered                : [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Parquet file                 : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_multiyear.parquet


,longitude,latitude,quality_flag,sensitivity,rh98,year,file_source,beam
0,-79.171967,38.044788,1,0.900244,2.92,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000
1,-79.171469,38.044455,1,0.916953,2.65,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000
2,-79.170971,38.044123,1,0.918692,2.65,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000
3,-79.170475,38.043790,1,0.912793,20.48,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000
4,-79.169977,38.043457,1,0.907017,29.40,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000


In [8]:
#Save to CSV locally to support Canopy Height Change
df_gedi.to_csv(OUTPUT_DETAILED_COUNTY_CSV, index=False)
print(f"Saved locally to : {OUTPUT_DETAILED_COUNTY_CSV}")

#df_gedi.to_csv('/home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_height.csv', index=False)
#print(f"Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_height.csv")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_height.csv


## Cell 8 — Phase 1b: Harmonize to the SMAP 9 km Grid

In [9]:
lon_bins = np.arange(MIN_LON, MAX_LON, GRID_RES)
lat_bins = np.arange(MIN_LAT, MAX_LAT, GRID_RES)

df_gedi['lon_grid'] = pd.cut(df_gedi['longitude'], bins=lon_bins, labels=lon_bins[:-1]).astype(float)
df_gedi['lat_grid'] = pd.cut(df_gedi['latitude'],  bins=lat_bins, labels=lat_bins[:-1]).astype(float)

gedi_grid = df_gedi.groupby(['year', 'lat_grid', 'lon_grid'])['rh98'].mean().reset_index()

ds_gedi = gedi_grid.set_index(['year', 'lat_grid', 'lon_grid']).to_xarray()
ds_gedi.to_netcdf(OUTPUT_NETCDF)

print(f"Grid cells produced : {len(gedi_grid):,}")
print(f"NetCDF saved to     : {OUTPUT_NETCDF}")
gedi_grid.head()

Grid cells produced : 1,610
NetCDF saved to     : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_canopy_grid.nc


,year,lat_grid,lon_grid,rh98
0,2019,37.3296,-79.1721,16.509153
1,2019,37.3296,-79.0911,16.554823
2,2019,37.3296,-79.0101,13.830095
3,2019,37.3296,-78.9291,12.423847
4,2019,37.3296,-78.8481,13.842674


## Cell 9 — Phase 1c: Fetch Jurisdiction Boundaries from US Census TIGERweb

In [10]:
boundary_gdfs = []

for name, st_fips, geo_id, geo_type in TARGET_JURISDICTIONS:
    try:
        b_gdf = fetch_boundary(name, st_fips, geo_id, geo_type)
        boundary_gdfs.append(b_gdf)
    except Exception as e:
        print(f"  Failed to fetch boundary for {name}: {e}")

if not boundary_gdfs:
    raise RuntimeError("No jurisdiction boundaries fetched. Cannot perform spatial join.")

filtered_counties = pd.concat(boundary_gdfs, ignore_index=True)
print(f"\nBoundaries fetched: {len(filtered_counties)} jurisdiction(s)")
filtered_counties[['jurisdiction', 'geometry']].head(10)

  Fetching boundary for Albemarle...
  Fetching boundary for Augusta...
  Fetching boundary for Buckingham...
  Fetching boundary for Charlottesville...
  Fetching boundary for Fluvanna...
  Fetching boundary for Greene...
  Fetching boundary for Louisa...
  Fetching boundary for Nelson...
  Fetching boundary for Rockingham...

Boundaries fetched: 9 jurisdiction(s)


,jurisdiction,geometry
0,Albemarle,"POLYGON ((-78.64831 37.73272, -78.64824 37.732..."
1,Augusta,"POLYGON ((-78.93069 38.31342, -78.93242 38.314..."
2,Buckingham,"POLYGON ((-78.24812 37.64089, -78.248 37.64158..."
3,Charlottesville,"POLYGON ((-78.52368 38.02233, -78.52371 38.022..."
4,Fluvanna,"POLYGON ((-78.15905 37.74933, -78.15889 37.749..."
5,Greene,"POLYGON ((-78.3587 38.29019, -78.35879 38.2904..."
6,Louisa,"POLYGON ((-78.30676 38.00647, -78.30687 38.006..."
7,Nelson,"POLYGON ((-78.90797 37.97698, -78.90798 37.976..."
8,Rockingham,"POLYGON ((-79.09295 38.66441, -79.09295 38.664..."


## Cell 10 — Phase 1c: Spatial Join and County-Level Aggregation

In [11]:
gdf_gedi = gpd.GeoDataFrame(
    df_gedi,
    geometry=gpd.points_from_xy(df_gedi.longitude, df_gedi.latitude),
    crs="EPSG:4326"
)

print("Performing spatial join...")
gedi_with_county = gpd.sjoin(
    gdf_gedi,
    filtered_counties[['jurisdiction', 'geometry']],
    how='inner',
    predicate='within'
)

gedi_county_summary = (
    gedi_with_county
    .groupby(['year', 'jurisdiction'])['rh98']
    .mean()
    .reset_index()
    .rename(columns={'rh98': 'canopy_height_mean_m'})
)

print(f"Spatial join complete. Rows in summary: {len(gedi_county_summary)}")
gedi_county_summary

Performing spatial join...
Spatial join complete. Rows in summary: 61


,year,jurisdiction,canopy_height_mean_m
0,2019,Albemarle,19.675812
1,2019,Augusta,14.742975
2,2019,Buckingham,16.031614
3,2019,Charlottesville,16.583229
4,2019,Fluvanna,18.198132
...,...,...,...
56,2025,Fluvanna,15.802814
57,2025,Greene,14.136559
58,2025,Louisa,15.712695
59,2025,Nelson,17.517622


---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
Cell In[14], line 11
      4 gdf_gedi_canopy_height = gpd.GeoDataFrame(
      5      df_gedi_canopy_height,
      6      geometry=gpd.points_from_xy(df_gedi_canopy_height.longitude, df_gedi_canopy_height.latitude),
      7      crs="EPSG:4326"
      8  )
     10 print("Performing spatial join...")
---> 11 gedi_with_county_canopy_height = gpd.sjoin(
     12     gdf_gedi_canopy_height,
     13     filtered_counties[['jurisdiction', 'geometry']],
     14     how='inner',
     15     predicate='within'
     16 )
     18 # gedi_county_summary = (
     19 #     gedi_with_county
     20 #     .groupby(['year', 'jurisdiction'])['rh98']
   (...)
     23 #     .rename(columns={'rh98': 'canopy_height_mean_m'})
     24 # )
     26 print(f"Spatial join complete. Rows in summary: {len(gedi_with_county_canopy_height)}")

File ~/anaconda3/envs/python3/lib/python3.10/site-packages/geopandas/tools/sjoin.py:119, in sjoin(left_df, right_df, how, predicate, lsuffix, rsuffix, distance, on_attribute, **kwargs)
    113 _basic_checks(left_df, right_df, how, lsuffix, rsuffix, on_attribute=on_attribute)
    115 indices = _geom_predicate_query(
    116     left_df, right_df, predicate, distance, on_attribute=on_attribute
    117 )
--> 119 joined, _ = _frame_join(
    120     left_df,
    121     right_df,
    122     indices,
    123     None,
    124     how,
    125     lsuffix,
    126     rsuffix,
    127     predicate,
    128     on_attribute=on_attribute,
    129 )
    131 return joined

File ~/anaconda3/envs/python3/lib/python3.10/site-packages/geopandas/tools/sjoin.py:461, in _frame_join(left_df, right_df, indices, distances, how, lsuffix, rsuffix, predicate, on_attribute)
    459 right_nlevels = right_df.index.nlevels
    460 right_index_original = right_df.index.names
--> 461 right_df, right_column_names = _reset_index_with_suffix(right_df, rsuffix, left_df)
    463 # if conflicting names in left and right, add suffix
    464 left_column_names, right_column_names = _process_column_names_with_suffix(
    465     left_column_names,
    466     right_column_names,
   (...)
    469     right_df,
    470 )

File ~/anaconda3/envs/python3/lib/python3.10/site-packages/geopandas/tools/sjoin.py:280, in _reset_index_with_suffix(df, suffix, other)
    278         # check new label will not be in other dataframe
    279         if new_label in df.columns or new_label in other.columns:
--> 280             raise ValueError(
    281                 f"'{new_label}' cannot be a column name in the frames being joined"
    282             )
    283         column_names[i] = new_label
    284 return df_reset, pd.Index(column_names)

ValueError: 'index_right' cannot be a column name in the frames being joined

In [15]:
# Load dataframe with canopy height data points
df_gedi_canopy_height = pd.read_csv('/home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_canopy_height.csv')

gdf_gedi_canopy_height = gpd.GeoDataFrame(
     df_gedi_canopy_height,
     geometry=gpd.points_from_xy(df_gedi_canopy_height.longitude, df_gedi_canopy_height.latitude),
     crs="EPSG:4326"
 )

# The Fix for the Value Error

# Drop any stale index columns from a previous join
gdf_gedi_canopy_height = gdf_gedi_canopy_height.drop(
    columns=[c for c in ["index_right", "index_left"] if c in gdf_gedi_canopy_height.columns]
)
filtered_counties = filtered_counties.drop(
    columns=[c for c in ["index_right", "index_left"] if c in filtered_counties.columns]
)

print("Performing spatial join...")
gedi_with_county_canopy_height = gpd.sjoin(
    gdf_gedi_canopy_height,
    filtered_counties[['jurisdiction', 'geometry']],
    how='inner',
    predicate='within'
)

# gedi_county_summary = (
#     gedi_with_county
#     .groupby(['year', 'jurisdiction'])['rh98']
#     .mean()
#     .reset_index()
#     .rename(columns={'rh98': 'canopy_height_mean_m'})
# )

print(f"Spatial join complete. Rows in summary: {len(gedi_with_county_canopy_height)}")
gedi_with_county_canopy_height.head(20)

Performing spatial join...
Spatial join complete. Rows in summary: 993671


,longitude,latitude,quality_flag,sensitivity,rh98,year,file_source,beam,geometry,jurisdiction_left,index_right,jurisdiction_right
0,-79.171967,38.044788,1,0.900244,2.92,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.17197 38.04479),Augusta,1,Augusta
1,-79.171469,38.044455,1,0.916953,2.65,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.17147 38.04446),Augusta,1,Augusta
2,-79.170971,38.044123,1,0.918692,2.65,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.17097 38.04412),Augusta,1,Augusta
3,-79.170475,38.043790,1,0.912793,20.48,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.17047 38.04379),Augusta,1,Augusta
4,-79.169977,38.043457,1,0.907017,29.40,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16998 38.04346),Augusta,1,Augusta
5,-79.169478,38.043124,1,0.961559,28.69,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16948 38.04312),Augusta,1,Augusta
6,-79.168979,38.042791,1,0.961136,7.79,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16898 38.04279),Augusta,1,Augusta
7,-79.168481,38.042459,1,0.915506,25.69,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16848 38.04246),Augusta,1,Augusta
8,-79.167985,38.042126,1,0.909027,24.83,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16798 38.04213),Augusta,1,Augusta
9,-79.167488,38.041793,1,0.921234,3.07,2019,GEDI02_A_2019113174925_O02048_03_T02621_02_003...,BEAM0000,POINT (-79.16749 38.04179),Augusta,1,Augusta


In [16]:
#Save to CSV to support County Canopy Height Change
gedi_with_county_canopy_height.to_csv(OUTPUT_DETAILED_COUNTY_CANOPY_HEIGHT_CSV, index=False)
print(f"Saved locally to : {OUTPUT_DETAILED_COUNTY_CANOPY_HEIGHT_CSV}")
#gedi_with_county_canopy_height.to_csv('/home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_canopy_height.csv', index=False)
#print(f"Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_canopy_height.csv")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_canopy_height.csv


## Cell 11 — Save County Summary Locally and Upload to S3

In [17]:
# Save locally
gedi_county_summary.to_csv(OUTPUT_COUNTY_CSV, index=False)
print(f"Saved locally to : {OUTPUT_COUNTY_CSV}")

# Upload to S3 for downstream pipeline consumption
county_csv_filename = os.path.basename(OUTPUT_COUNTY_CSV)
s3_county_key       = GEDI_COUNTY_S3_PREFIX + county_csv_filename
s3_client           = boto3.client("s3")

s3_client.upload_file(
    OUTPUT_COUNTY_CSV,
    S3_BUCKET,
    s3_county_key,
    ExtraArgs={"ContentType": "text/csv"}
)

s3_county_uri = f"s3://{S3_BUCKET}/{s3_county_key}"
print(f"Uploaded to S3   : {s3_county_uri}")
print("\nAll phases completed successfully!")

Saved locally to : /home/ec2-user/SageMaker/gedi_output/virginia_gedi_county_summary.csv
Uploaded to S3   : s3://central-virginia-tree-canopy-project/gedi-county-summary/virginia_gedi_county_summary.csv

All phases completed successfully!


## Cell 12 — Save Detailed County locally and Upload to S3

In [ ]:
# Save detailed county output to S3
gedi_with_county_canopy_height.to_csv(OUTPUT_DETAILED_COUNTY_CANOPY_HEIGHT_CSV, index=False)
print(f"Saved locally to : {OUTPUT_DETAILED_COUNTY_CANOPY_HEIGHT_CSV}")

# Upload to S3 for downstream pipeline consumption
county_csv_filename = os.path.basename(OUTPUT_DETAILED_COUNTY_CANOPY_HEIGHT_CSV)
s3_county_key       = GEDI_DETAILED_COUNTY_S3_PREFIX + county_csv_filename
s3_client           = boto3.client("s3")

s3_client.upload_file(
    OUTPUT_DETAILED_COUNTY_CANOPY_HEIGHT_CSV,
    S3_BUCKET,
    s3_county_key,
    ExtraArgs={"ContentType": "text/csv"}
)

s3_county_uri = f"s3://{S3_BUCKET}/{s3_county_key}"
print(f"Uploaded to S3   : {s3_county_uri}")
print("\nAll detailed phases completed successfully!")